# 02 · Laya: zero-shot base vs typed-decisions vs our fine-tune

**Laya** answers *typed questions* about a *state* in a single encoder forward pass (no text generation).
Our three output fields map straight onto its three question types:

| our field | Laya question type |
|---|---|
| `route_target` | `choice` (pick one of 6 teams) |
| `priority_level` | `score` (ordered levels low → critical) |
| `requires_human` | `noul` (yes/no, returns P(true)) |

We compare three Laya arms on the **same test set** as notebook 01:
1. `convaiinnovations/laya`: the general English base, **zero-shot** (never saw our data)
2. `convaiinnovations/laya-typed-decisions`: the authors' fine-tune on synthetic workflows (incl. customer service), **zero-shot** on our data
3. **our fine-tune**: base Laya trained on our train split with `laya.train.finetune` (RLCD loss)

**Runtime:** switch Colab to a **GPU** (Runtime → Change runtime type → T4). Each checkpoint is ~800 MB.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

working dir: /content/typed-decision-models


Install Laya. `USE_TF=0` stops `transformers` from probing TensorFlow (installed on Colab), which can hang model loading.

In [2]:
%pip install -q laya scikit-learn xgboost pandas
os.environ["USE_TF"] = "0"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 12.8 MB/s eta 0:00:00
device: cuda


## 1. Load the same splits as notebook 01

`load_splits` reads `data/*.csv`, or rebuilds the identical synthetic data (same seed) if this is a fresh runtime.

In [3]:
from src.dataset import load_splits, laya_questions, row_to_state

train_df, val_df, test_df = load_splits("data")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

train=2099  val=451  test=450


## 2. What Laya actually sees

The **state** is the ticket as a dict (text + metadata). The **questions** are the same three typed questions for every
ticket; the `criteria` are short descriptions of each option, which is all a zero-shot model has to go on.

In [4]:
import json

print(json.dumps(row_to_state(test_df.iloc[0]), indent=2))
print(json.dumps(laya_questions(), indent=2))

{
  "query": "please cancel order 40920 before it ships this is urgent",
  "account_tier": "free",
  "channel": "email",
  "urgency_score": 0.514,
  "past_failures": 1
}
{
  "route_target": {
    "type": "choice",
    "instructions": "Which support team should handle this ticket?",
    "criteria": {
      "billing": "payments, charges, refunds, invoices, subscriptions",
      "orders": "placing, changing, tracking or cancelling orders and deliveries",
      "account": "login details, profile changes, creating or deleting an account",
      "tech_support": "bugs, errors, crashes, outages, the app or site not working",
      "fraud": "unauthorised charges, hacked accounts, scams, suspicious activity",
      "general": "feedback, general questions, anything else"
    }
  },
  "priority_level": {
    "type": "score",
    "instructions": "How urgent is this ticket, given the customer's tier, urgency and past failures?",
    "criteria": [
      "low",
      "medium",
      "high",
      "cri

## 3. Zero-shot: base Laya

`LayaDecisionModel` (in `src/models/laya_model.py`) wraps any checkpoint behind the same `DecisionModel` interface as RF/XGBoost,
so the benchmark code is identical. Let's look at one raw prediction first.

In [5]:
from src.models.laya_model import LayaDecisionModel, BASE_MODEL_ID, TYPED_DECISIONS_MODEL_ID

laya_base = LayaDecisionModel(BASE_MODEL_ID, device=DEVICE).load()
raw = laya_base.agent.predict(row_to_state(test_df.iloc[0]), laya_questions())
print(json.dumps(raw["answers"], indent=2))
print("true labels:", test_df.iloc[0][["route_target", "priority_level", "requires_human"]].to_dict())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


/usr/local/lib/python3.13/dist-packages/laya/agent.py:2149: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


{
  "route_target": {
    "type": "choice",
    "choice": "orders",
    "probabilities": {
      "billing": 0.0033,
      "orders": 0.9738,
      "account": 0.0013,
      "tech_support": 0.0135,
      "fraud": 0.0048,
      "general": 0.0032
    },
    "confidence": 0.913,
    "answer_confidence": 0.9738,
    "action": {
      "act_probability": 1.0
    }
  },
  "priority_level": {
    "type": "score",
    "score": 1.8076,
    "legend": {
      "0": "low",
      "1": "medium",
      "2": "high",
      "3": "critical"
    },
    "probabilities": {
      "0": 0.2059,
      "1": 0.0743,
      "2": 0.4261,
      "3": 0.2937
    },
    "confidence": 0.1042,
    "answer_confidence": 0.4261,
    "action": {
      "act_probability": 1.0
    }
  },
  "requires_human": {
    "type": "noul",
    "noul": 0.9055,
    "confidence": 0.9055,
    "answer_confidence": 0.9055,
    "action": {
      "act_probability": 1.0
    }
  }
}
true labels: {'route_target': 'general', 'priority_level': 'medium', 're

Now the full test set + latency (p50/p95 over 100 single-ticket calls).

In [6]:
from src.interface import evaluate_model

results = {}
results["laya:laya"] = evaluate_model(laya_base, test_df, n_latency_runs=100)
results["laya:laya"]

{'model': 'laya:laya',
 'route_acc': 0.6355555555555555,
 'route_f1': 0.6145935810906203,
 'priority_acc': 0.31555555555555553,
 'priority_f1': 0.19011541148082298,
 'requires_acc': 0.7066666666666667,
 'requires_f1': 0.5297805642633229,
 'exact_match': 0.17333333333333334,
 'p50_ms': 45.738414499993496,
 'p95_ms': 50.395751150020374,
 'size_mb': 1607.1084098815918}

## 4. Zero-shot: `laya-typed-decisions`

Same architecture, but already fine-tuned by the authors on synthetic typed-decision workflows. Does that transfer to our
routing schema without any training on our data?

In [7]:
laya_td = LayaDecisionModel(TYPED_DECISIONS_MODEL_ID, device=DEVICE).load()
results["laya:laya-typed-decisions"] = evaluate_model(laya_td, test_df, n_latency_runs=100)

# free GPU memory before fine-tuning
del laya_td
torch.cuda.empty_cache() if torch.cuda.is_available() else None

/usr/local/lib/python3.13/dist-packages/laya/agent.py:2149: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


## 5. Fine-tune base Laya on our tickets

`finetune_laya` writes our rows as Laya JSONL (`{state, questions, expected}`) and calls the package's own trainer
`laya.train.finetune`. Its default loss is **RLCD**: the model is rewarded with a *proper scoring rule* (it scores well only
by putting high probability on the right answer), plus a soft cross-entropy term. After training it refits calibration
temperatures and saves a checkpoint folder that `laya.load` can read.

Knobs:
- `EPOCHS`: 2 is enough for ~2k tickets; raise if val accuracy is still climbing.
- `FREEZE_ENCODER=True`: train only the small decision head. Much faster / lighter, usually a bit less accurate.
- `SAVE_TO_DRIVE`: copy the checkpoint to Google Drive so notebook 03 (a new runtime) can load it.

In [8]:
from src.models.laya_model import finetune_laya

EPOCHS = 2
FREEZE_ENCODER = False
FINETUNED_DIR = "artifacts/laya-finetuned"

summary = finetune_laya(train_df, FINETUNED_DIR, base_model=BASE_MODEL_ID, val_df=val_df,
                        epochs=EPOCHS, micro_batch=8, grad_accum=4,
                        freeze_encoder=FREEZE_ENCODER, device=DEVICE)
summary

eval items 1353 from artifacts/laya-finetuned_data/val.jsonl (skipped {}, 0 overlapping training/calibration items)
train items 5897, calibration items 400, eval items 1353 (artifacts/laya-finetuned_data/val.jsonl), skipped {}, device cuda
optimizer budget: 5897 train item(s), effective batch up to 32 (micro_batch 8 x grad_accum 4), 370 optimizer updates over 2 epoch(s)


/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/2 step 100 loss 0.4179
epoch 1/2 step 200 loss 0.5067
epoch 1/2 step 300 loss 0.4660
epoch 1/2 step 400 loss 0.3280
epoch 1/2 step 500 loss 0.0306
epoch 1/2 step 600 loss 0.5542
epoch 1/2 step 700 loss 0.0575
epoch 1/2 mean loss 0.4920
epoch 2/2 step 100 loss 0.1672
epoch 2/2 step 200 loss 0.2684
epoch 2/2 step 300 loss 0.1415
epoch 2/2 step 400 loss 0.7847
epoch 2/2 step 500 loss 0.5227
epoch 2/2 step 600 loss 0.1915
epoch 2/2 step 700 loss -0.0935
epoch 2/2 mean loss 0.3855

=== Evaluation (before vs after fine-tuning on 1353 held-out items from artifacts/laya-finetuned_data/val.jsonl) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.5447         0.9113         +0.3666       
Loss                   1.0107         0.3347         -0.6760       
ECE                    0.1732         0.0145         -0.1587       
Brier (multi-class)    0.5724         0.1536         -0

{'train_items': 5897,
 'calibration_items': 400,
 'eval_items': 1353,
 'skipped': {},
 'epoch_loss': [0.49200403286386435, 0.38545093373875045],
 'temperature': [1.7169342041015625, 1.716921329498291, 2.2765121459960938],
 'temperature_by_options': {},
 'n_by_bucket': {'choice:6-10': 133, 'score:3-5': 124, 'noul:2': 143},
 'calibration': {'choice': {'items': 133,
   'temperature': 1.7169342041015625,
   'issues': [],
   'accuracy': 0.9548872180451128,
   'mean_confidence': 0.9529700905817866},
  'score': {'items': 124,
   'temperature': 1.716921329498291,
   'issues': [],
   'accuracy': 0.8629032258064516,
   'mean_confidence': 0.8755698461693308},
  'noul': {'items': 143,
   'temperature': 2.2765121459960938,
   'issues': [],
   'accuracy': 0.916083916083916,
   'mean_confidence': 0.9177524437375453}},
 'abstention_thresholds': {'choice:6-10': 0.6111,
  'score:3-5': 0.8062,
  'noul:2': 0.5992},
 'train_report': {'eval_source': 'artifacts/laya-finetuned_data/val.jsonl',
  'eval_mode': 

Optional: persist the fine-tuned checkpoint to Google Drive.

In [14]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(FINETUNED_DIR, "/content/drive/MyDrive/typed-decision-models/laya-finetuned", dirs_exist_ok=True)
    print("saved to Drive")

Mounted at /content/drive
saved to Drive


## 6. Evaluate our fine-tuned Laya

Loaded from the local folder, same wrapper, same test set.

In [10]:
del laya_base
torch.cuda.empty_cache() if torch.cuda.is_available() else None

laya_ft = LayaDecisionModel(FINETUNED_DIR, name="laya:finetuned", device=DEVICE).load()
results["laya:finetuned"] = evaluate_model(laya_ft, test_df, n_latency_runs=100)

## 7. Side by side

`exact_match` is the share of tickets where all three fields are right, which is what an agent router needs.

In [11]:
import pandas as pd

laya_results = pd.DataFrame(results.values()).set_index("model")
os.makedirs("artifacts", exist_ok=True)
laya_results.to_csv("artifacts/results_laya.csv")
laya_results.round(3)

,route_acc,route_f1,priority_acc,priority_f1,requires_acc,requires_f1,exact_match,p50_ms,p95_ms,size_mb
model,,,,,,,,,,
laya:laya,0.636,0.615,0.316,0.190,0.707,0.530,0.173,45.738,50.396,1607.108
laya:laya-typed-decisions,0.742,0.674,0.453,0.302,0.689,0.574,0.296,47.519,53.258,1607.108
laya:finetuned,0.958,0.958,0.791,0.679,0.936,0.916,0.720,55.985,59.751,1607.108


## 8. Is the confidence trustworthy?

The model card warns Laya is **over-confident**. A quick check: bucket test tickets by the route confidence and compare
it with the actual route accuracy in each bucket. If the model is well calibrated, the two columns should be close.

In [12]:
from src.interface import outputs_to_frame

outs = laya_ft.predict_batch(test_df)
check = pd.DataFrame({
    "confidence": [o.confidence for o in outs],
    "correct": (outputs_to_frame(outs)["route_target"].values == test_df["route_target"].values),
})
check["bucket"] = pd.cut(check["confidence"], [0, 0.5, 0.7, 0.9, 1.0])
check.groupby("bucket", observed=True).agg(mean_confidence=("confidence", "mean"),
                                           accuracy=("correct", "mean"), n=("correct", "size")).round(3)

,mean_confidence,accuracy,n
bucket,,,
"(0.9, 1.0]",0.956,0.958,450


**Takeaways to fill in after running:**
- Zero-shot base vs typed-decisions vs fine-tuned exact match: _TODO_
- How much did fine-tuning help priority (the field that depends most on metadata)? _TODO_
- GPU latency p50/p95: _TODO_

Next: `03_full_benchmark_laya_vs_xgboost_rf_bert.ipynb`.